In [1]:
import sqlite3
import json

# --- 1. CONFIGURACIÓN DE BASE DE DATOS SQL ---
def inicializar_base_datos():
    conn = sqlite3.connect(':memory:') # Base de datos SQL en memoria para la demo
    cursor = conn.cursor()
    
    # Crear tabla de feedback de clientes
    cursor.execute('''
        CREATE TABLE customer_feedback (
            id INTEGER PRIMARY KEY,
            cliente TEXT,
            comentario TEXT,
            sentimiento TEXT,
            respuesta_ia TEXT
        )
    ''')
    
    # Insertar datos de prueba (Simulación de entrada)
    feedback_datos = [
        ("Carlos M.", "El servicio de entrega tardó 3 días más de lo esperado. Muy molesto."),
        ("Laura G.", "¡Excelente atención al cliente y el producto llegó impecable! Volvería a comprar."),
        ("Diego R.", "El producto funciona bien, pero el empaque llegó un poco golpeado.")
    ]
    cursor.executemany("INSERT INTO customer_feedback (cliente, comentario) VALUES (?, ?)", feedback_datos)
    conn.commit()
    return conn

# --- 2. MOTOR DE IA PARA ANÁLISIS Y RESPUESTA ---
def procesar_comentario_con_ia(comentario):
    """
    Simula el análisis de sentimiento y generación de respuesta usando IA (OpenAI / Claude / Gemini).
    """
    comentario_lower = comentario.lower()
    if "excelente" in comentario_lower or "impecable" in comentario_lower:
        sentimiento = "POSITIVO"
        respuesta = "¡Muchas gracias por tu valoración! Nos alegra saber que tuviste una gran experiencia."
    elif "molesto" in comentario_lower or "tardó" in comentario_lower:
        sentimiento = "NEGATIVO"
        respuesta = "Lamentamos mucho la demora. Estamos revisando con el equipo de logística para que no vuelva a ocurrir."
    else:
        sentimiento = "NEUTRO"
        respuesta = "Agradecemos tus comentarios. Tomamos nota sobre el empaque para seguir mejorando."
        
    return sentimiento, respuesta

# --- 3. PIPELINE COMPLETO DE AUTOMATIZACIÓN ---
if __name__ == "__main__":
    print("\n========================================================")
    print("  AI & SQL PIPELINE: AUTOMATIZACIÓN DE FEEDBACK DE CLIENTES")
    print("========================================================\n")
    
    # Conectar a la BD
    conexion = inicializar_base_datos()
    cursor = conexion.cursor()
    
    # Obtener comentarios pendientes de procesar
    cursor.execute("SELECT id, cliente, comentario FROM customer_feedback")
    registros = cursor.fetchall()
    
    print(f"--> Se encontraron {len(registros)} comentarios en la Base de Datos SQL.\n")
    
    # Procesar cada registro con la IA y actualizar la BD
    for reg in registros:
        reg_id, cliente, comentario = reg
        print(f"Procesando ID {reg_id} | Cliente: {cliente}")
        
        # Llamada al módulo de IA
        sentimiento, respuesta_ia = procesar_comentario_con_ia(comentario)
        
        # Actualizar la fila en SQL con los insights de la IA
        cursor.execute('''
            UPDATE customer_feedback 
            SET sentimiento = ?, respuesta_ia = ? 
            WHERE id = ?
        ''', (sentimiento, respuesta_ia, reg_id))
    
    conexion.commit()
    print("\n--> ¡Procesamiento completado y guardado en SQL exitosamente!\n")
    
    # --- 4. CONSULTA Y MOSTRAR RESULTADOS FINALES DE LA BD ---
    print("--- RESULTADOS EN LA BASE DE DATOS (TABLA ACTUALIZADA) ---\n")
    cursor.execute("SELECT id, cliente, sentimiento, respuesta_ia FROM customer_feedback")
    resultados = cursor.fetchall()
    
    for r in resultados:
        print(f"ID: {r[0]} | Cliente: {r[1]} | Sentimiento IA: [{r[2]}]")
        print(f"   Respuesta Generada: \"{r[3]}\"\n")
        
    conexion.close()


  AI & SQL PIPELINE: AUTOMATIZACIÓN DE FEEDBACK DE CLIENTES

--> Se encontraron 3 comentarios en la Base de Datos SQL.

Procesando ID 1 | Cliente: Carlos M.
Procesando ID 2 | Cliente: Laura G.
Procesando ID 3 | Cliente: Diego R.

--> ¡Procesamiento completado y guardado en SQL exitosamente!

--- RESULTADOS EN LA BASE DE DATOS (TABLA ACTUALIZADA) ---

ID: 1 | Cliente: Carlos M. | Sentimiento IA: [NEGATIVO]
   Respuesta Generada: "Lamentamos mucho la demora. Estamos revisando con el equipo de logística para que no vuelva a ocurrir."

ID: 2 | Cliente: Laura G. | Sentimiento IA: [POSITIVO]
   Respuesta Generada: "¡Muchas gracias por tu valoración! Nos alegra saber que tuviste una gran experiencia."

ID: 3 | Cliente: Diego R. | Sentimiento IA: [NEUTRO]
   Respuesta Generada: "Agradecemos tus comentarios. Tomamos nota sobre el empaque para seguir mejorando."

